# HUM-450 Preliminary Analysis
The goal of this notebooks is to develop and test analysis tool for the project

In [ ]:
import numpy as np
import pandas as pd
import os
import re
from collections import Counter, defaultdict
import networkx as nx
import folium
from folium.plugins import MarkerCluster
import geopy


#Change the root path
ROOT="/Users/edouardkoehn/Documents/Github/HUM-450"
path_data=ROOT+"/data"

## Person Analysis

In [ ]:
def load_database(path):
    """
    Simple method for loading the database
    """
    database = pd.read_csv(path, sep=';')
    database['persons_mentioned']=database['persons_mentioned'].apply(lambda x: x[1:-1].replace("'", '').replace(", ",",").split(','))
    database['locations_mentioned']=database['locations_mentioned'].apply(lambda x: x[1:-1].replace("'", '').replace(", ",",").split(','))
    return database

def get_personality_dist(data:pd.DataFrame):
    """
    Compute the distribution of personalities mentioned in the data.

    Parameters:
    - data (pd.DataFrame): DataFrame containing the column 'persons_mentioned' listing the personalities mentioned.

    Returns:
    - annuary (defaultdict): Dictionary containing the personality distribution, where each personality is associated with its probability.
    """
    temp = []
    for persons in data['persons_mentioned']:
        temp = np.concatenate([temp, persons], axis=0)
    annuary = Counter(temp)
    size_annuary = [word_count for word, word_count in annuary.items()]
    size_annuary = np.sum(size_annuary)
    annuary = defaultdict(float, {word: word_count/size_annuary for word, word_count in annuary.items()})
        
    return annuary

def top_k_annuary(annuary:dict, k:int):
    """
    Return the top k personalities from the given annuary.

    Parameters:
    - annuary (dict): Dictionary containing the personality distribution.
    - k (int): Number of top personalities to return.

    Returns:
    - top_k (list): List of tuples containing the top k personalities and their probabilities.
    """
    return sorted(annuary.items(), key=lambda item: item[1], reverse=True)[:k]

def get_adjacency_matrix_personnality(annuary:dict, list_person_mentioned:np.array, k:int):
    """
    Generate the adjacency matrix representing relationships between personalities.

    Parameters:
    - annuary (dict): Dictionary containing the personality distribution.
    - list_person_mentioned (np.array): Array of articles mentioning personalities.
    - k (int): Number of nodes in the graph.

    Returns:
    - df_graph (pd.DataFrame): Adjacency matrix representing relationships between personalities.
    """
    nodes = []
    for person in top_k_annuary(annuary, k):
        nodes.append(person[0])

    adjacency = np.zeros((len(nodes), len(nodes)))
    for node, x in zip(nodes, np.arange(0, len(nodes))):
        for article in list_person_mentioned:
            if node in article:
                for target, y in zip(nodes, np.arange(0, len(nodes))):
                    if (target in article) & (node != target):
                        adjacency[x, y] += 1
    df_graph = pd.DataFrame(adjacency, index=nodes, columns=nodes)
    return df_graph

    

In [ ]:
#Load the data
extraction=load_database(path_data+"/database_1890-1910.csv")
extraction.head()

In [ ]:
annuary=get_personality_dist(extraction)
top_k_annuary(annuary, 10)

In [ ]:
df_graph=get_adjacency_matrix_personnality(annuary,extraction['persons_mentioned'].to_numpy(),10)
G=nx.from_pandas_adjacency(df_graph)
nx.draw_networkx(G)